In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import sys
import os
from pathlib import Path

In [13]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [14]:
sys.path.append(root_path)

In [19]:
from wilsonelser.transformation.data_transformer import DataTransformer
from wilsonelser.transformation.base_data_transformer import BaseDataTransformer
from wilsonelser.transformation.table_writer import TableWriter
from wilsonelser.transformation.table_reader import TableReader

In [59]:
config_file = "use_cases/p2c_insightops/sales_order_invoice_consolidated.yml"
table_reader = TableReader(config_file)
transformer = BaseDataTransformer(config_file)
transformer2 = DataTransformer(config_file)
writer = TableWriter(config_file)

In [49]:
df_ibm = transformer2.apply_joins("ibm")

INFO:adm.integration.base_integration:Applying joins for combine_id: ibm
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: ibmdb2_sales_order_invoice_max with alias: m
INFO:adm.integration.base_integration:Starting transformations for ID: max_inserted_timestamp
INFO:adm.integration.base_integration:Joining i with m using condition: i.__inserted_timestamp = m.max_inserted_tsp and join type: inner


INFO:adm.integration.base_integration:Schema after joining i with m: StructType([StructField('shipment_date', DateType(), True), StructField('order_date', DateType(), True), StructField('invoice_number', StringType(), True), StructField('total_invoice_amount_usd', DoubleType(), True), StructField('invoice_date', DateType(), True), StructField('payment_term_code', StringType(), True), StructField('entity_document_company_code', StringType(), True), StructField('entity_document_company_name', StringType(), True), StructField('user_id', StringType(), True), StructField('payment_term_description', StringType(), True), StructField('invoice_due_date', DateType(), True), StructField('sold_to_customer_no', IntegerType(), True), StructField('document_type', StringType(), True), StructField('sold_to_customer_name', StringType(), True), StructField('source_system', StringType(), True), StructField('currency_code', StringType(), True), StructField('region', StringType(), True), StructField('segmen

In [54]:
df_jde = transformer2.apply_joins("jde")

INFO:adm.integration.base_integration:Applying joins for combine_id: jde
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: jdexeeu_sales_order_invoice_max with alias: m
INFO:adm.integration.base_integration:Starting transformations for ID: max_inserted_timestamp
INFO:adm.integration.base_integration:Joining i with m using condition: i.__inserted_timestamp = m.max_inserted_tsp and join type: inner


INFO:adm.integration.base_integration:Schema after joining i with m: StructType([StructField('shipment_date', DateType(), True), StructField('order_date', DateType(), True), StructField('invoice_number', IntegerType(), True), StructField('total_invoice_amount_usd', DoubleType(), True), StructField('invoice_date', DateType(), True), StructField('payment_term_code', StringType(), True), StructField('entity_document_company_code', IntegerType(), True), StructField('entity_document_company_name', StringType(), True), StructField('user_id', StringType(), True), StructField('payment_term_description', StringType(), True), StructField('invoice_due_date', DateType(), True), StructField('sold_to_customer_no', IntegerType(), True), StructField('document_type', StringType(), True), StructField('sold_to_customer_name', StringType(), True), StructField('source_system', StringType(), True), StructField('currency_code', StringType(), True), StructField('region', StringType(), True), StructField('segm

In [55]:
df_u = df_ibm.unionByName(df_jde, allowMissingColumns=True)

In [60]:
df_u = transformer.apply_transformations("add_tsp", df_u)

INFO:adm.integration.base_integration:Starting transformations for ID: add_tsp


In [63]:
spark.conf.set("spark.sql.legacy.charVarcharAsString", True)
writer.write_table("sales_order_invoice", df_u)

INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.sales_order_invoice: 34896417
